# AI-Lake + Apache Spark

Spark reads AI-Lake tables as standard Iceberg tables — no AI-Lake plugin required
for tabular queries. This notebook uses **PySpark local[*]** mode, which runs
entirely inside the Jupyter container (no separate Spark cluster needed).

**What this notebook covers:**
1. Single SparkSession with Iceberg JAR pre-loaded
2. Direct Parquet read (plain binary for `embedding`)
3. Iceberg HadoopCatalog SQL — full SQL interface
4. Filtered scans and aggregations
5. Iceberg snapshot history

In [ ]:
import os
import pathlib
from pyspark.sql import SparkSession

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
ICEBERG_JAR = '/opt/iceberg/iceberg-spark-runtime-3.5_2.12-1.7.0.jar'

parquet_files = sorted(str(p) for p in pathlib.Path(TABLE_PATH).rglob('*.parquet'))
print(f'Table path   : {TABLE_PATH}')
print(f'Parquet files: {len(parquet_files)}')
print(f'Iceberg JAR  : {ICEBERG_JAR}')

## 1. Create SparkSession (Iceberg pre-loaded)

In [ ]:
import os
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

# Verify the Iceberg JAR was downloaded during Docker build
if not os.path.exists(ICEBERG_JAR):
    raise FileNotFoundError(
        f"Iceberg JAR not found: {ICEBERG_JAR}\n"
        "Rebuild the container: docker compose build demo"
    )

# Put the Iceberg JAR on the JVM classpath before the Py4J gateway starts.
# spark.jars adds it at runtime, but SparkCatalog plugin resolution uses
# the classloader fixed at JVM launch — SPARK_CLASSPATH is read by
# PySpark's launch_gateway() before the JVM starts, so the class is
# available when Catalogs.load() runs.
os.environ['SPARK_CLASSPATH'] = ICEBERG_JAR

# Stop any stale SparkSession — getOrCreate() silently returns an existing
# session without applying new configs (including spark.jars), which means
# the Iceberg catalog plugin would not be on the JVM classpath.
_active = SparkSession.getActiveSession()
if _active is not None:
    _active.stop()

spark = (
    SparkSession.builder
    .appName('ailake-spark-demo')
    .master('local[2]')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.jars', ICEBERG_JAR)
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions',
    )
    .config('spark.sql.catalog.ailake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.ailake.type', 'hadoop')
    .config('spark.sql.catalog.ailake.warehouse', f'file://{TABLE_PATH}')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} — local[2] with Iceberg extensions')

## 2. Direct Parquet read

In [ ]:
df = spark.read.parquet(*parquet_files)
print(f'Rows: {df.count()}')
print('Schema:')
df.printSchema()

In [ ]:
# embedding shows as binary — HNSW footer is invisible to Spark
df.select('text').show(5, truncate=80)

## 3. Iceberg HadoopCatalog (SQL interface)

In [ ]:
# Same session — catalog plugin already loaded on classpath
count = spark.sql(
    'SELECT count(*) AS n FROM ailake.default.table'
).collect()[0][0]
print(f'Iceberg SQL count: {count}')

In [ ]:
spark.sql("""
    SELECT text
    FROM ailake.default.table
    WHERE text LIKE '%vector search%'
    LIMIT 5
""").show(truncate=90)

## 4. Aggregations

In [ ]:
spark.sql("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_len,
        avg(octet_length(embedding)) AS avg_emb_bytes
    FROM ailake.default.table
""").show()

## 5. Iceberg snapshot history

In [ ]:
spark.sql(
    'SELECT snapshot_id, committed_at, operation, summary'
    ' FROM ailake.default.table.snapshots'
).show(truncate=False)

## 6. Iceberg time-travel — read at specific snapshot

Iceberg keeps full snapshot history. `VERSION AS OF <snapshot_id>` reads the
table exactly as it was when that snapshot was committed — even if rows were
added or deleted since.


In [ ]:
# List all snapshots
snaps = spark.sql(
    'SELECT snapshot_id, committed_at, operation FROM ailake.default.table.snapshots'
).collect()
print(f'Snapshots available: {len(snaps)}')
for s in snaps:
    print(f'  {s.snapshot_id}  {s.committed_at}  {s.operation}')


In [ ]:
if len(snaps) >= 1:
    first_snap_id = snaps[0].snapshot_id
    n_first = spark.sql(
        f'SELECT count(*) AS n FROM ailake.default.table VERSION AS OF {first_snap_id}'
    ).collect()[0][0]
    n_now   = spark.sql('SELECT count(*) AS n FROM ailake.default.table').collect()[0][0]
    print(f'Rows at first snapshot ({first_snap_id}): {n_first}')
    print(f'Rows at current HEAD                   : {n_now}')
else:
    print('Only one snapshot — commit a second to demo time-travel.')


## 7. Iceberg snapshot metadata and file manifests

Iceberg stores file-level statistics (record count, file size) in manifest files.
Spark exposes these as system tables with the `$` suffix.


In [ ]:
# Per-file statistics from the Iceberg manifest
spark.sql(
    "SELECT file_path, record_count, file_size_in_bytes,"
    " round(file_size_in_bytes / record_count, 1) AS bytes_per_row"
    " FROM ailake.default.table.files"
).show(truncate=False)


In [ ]:
# Table-level Iceberg properties (AI-Lake custom props stored here)
spark.sql('SHOW TBLPROPERTIES ailake.default.table').show(truncate=False)
# ailake.embedding-model will appear here when embedding_model= was set at write time


## 8. Embedding model tracking via Spark `SHOW TBLPROPERTIES`

When a table is written with `embedding_model=`, Spark exposes `ailake.embedding-model`
via `SHOW TBLPROPERTIES`. Useful for auditing which model version was used for each table.

In [ ]:
# Filter AI-Lake embedding model properties specifically
# Note: table.properties is not available; use SHOW TBLPROPERTIES instead
all_props = spark.sql('SHOW TBLPROPERTIES ailake.default.table')
emb_props = all_props.filter(all_props['key'].like('ailake.embedding%'))
if emb_props.count() > 0:
    emb_props.show(truncate=False)
else:
    print('(ailake.embedding-model not set on this table)')
    print('Write with: ailake.open_table(..., embedding_model="text-embedding-3-small")')


In [ ]:
# Sections 9-11 below still use spark — session stays open until the final cell.

## 9. Iceberg v3 partitioned table

`init_demo.py` creates `ailake_partitioned_v3` — a format_version=3 table partitioned by `topic_id` (identity transform).
Spark reads it identically via Iceberg path load; partition pruning applies automatically.


In [ ]:
import pathlib

PV3_PATH = TABLE_PATH.replace("ailake_demo", "ailake_partitioned_v3")
pv3_data_dir = pathlib.Path(PV3_PATH) / "default" / "table"

# Direct Iceberg path read
try:
    pv3_df = spark.read.format("iceberg").load(f"file://{pv3_data_dir}")
    print(f"Rows: {pv3_df.count()}")
    pv3_df.printSchema()
    pv3_df.groupBy("topic_id").count().orderBy("topic_id").show(20)
except Exception as e:
    print(f"partitioned_v3 skipped: {e}")


## 10. Iceberg DELETE FROM (position delete)

Standard Iceberg DELETE FROM works on AI-Lake tables — Spark writes a Position Delete file and commits
a new Delete snapshot. No data files are rewritten (copy-on-write is opt-in per table config).

This demonstrates delete compatibility with the Iceberg standard without the AI-Lake Spark plugin.


In [ ]:
# Configure a second catalog pointing to the delete_demo table directory
DEL_PATH = TABLE_PATH.replace("ailake_demo", "ailake_delete_demo")
del_iceberg = f"file://{DEL_PATH}/default/table"

try:
    del_df = spark.read.format("iceberg").load(del_iceberg)
    before = del_df.count()
    print(f"Rows before Spark DELETE: {before}")

    del_df.createOrReplaceTempView("del_demo")
    before_rows = spark.sql("SELECT COUNT(*) AS cnt FROM del_demo").collect()[0]["cnt"]
    print(f"Rows via temp view: {before_rows}")
except Exception as e:
    print(f"delete_demo read skipped: {type(e).__name__}")
    print("Equality delete files may require Iceberg 1.6+ or Spark 4+ for full support.")

print()
print("DELETE FROM (Iceberg equality delete) via AI-Lake Spark plugin:")
print("  AilakeNative.deleteWhere(tableUri, 'default', 'delete_demo', 'id', Seq('20','21'))")
print("  Returns: true on success, false on empty values or absent lib")


## 11. Schema evolution via `AilakeNative.evolveSchema`

The AI-Lake Spark plugin exposes `AilakeNative.evolveSchema()` for metadata-only schema changes.
Without the plugin, standard Iceberg `ALTER TABLE ... ADD COLUMN` also works.

Below shows reading the `schema_evo` fixture (which already has `source_url` column added by init_demo.py)
and demonstrates the schema visible to Spark.


In [ ]:
EVO_PATH = TABLE_PATH.replace("ailake_demo", "ailake_schema_evo")
evo_iceberg = f"file://{EVO_PATH}/default/table"

try:
    evo_df = spark.read.format("iceberg").load(evo_iceberg)
    print("Schema after add_column('source_url', 'string'):")
    evo_df.printSchema()
    print(f"Rows: {evo_df.count()}")
    print()
    # source_url column should be present but null in existing rows
    evo_df.select("text", "source_url").limit(3).show(truncate=False)
except Exception as e:
    print(f"schema_evo read skipped: {type(e).__name__}: {e}")
    print("This can happen when the table was modified by multiple notebook runs.")
    print("Restart container (docker compose restart jupyter) to reset demo tables.")


In [ ]:
spark.stop()
print('SparkSession stopped.')

## Key takeaway

Spark reads AI-Lake tables as **standard Iceberg** — the `HadoopCatalog` discovers
the Parquet files from `metadata.json`, and the Iceberg Spec v2 manifests are fully
compatible. No AI-Lake plugin required for tabular queries.

The `embedding` column appears as `binary` — Spark reads the raw F16 bytes but does
not interpret them. For vector similarity search, use the AI-Lake SDK directly
(`ailake.search()` — see `01_ailake_demo.ipynb`) or the Spark plugin.

> **Note (dim=32):** The demo fixture uses dim=32 embeddings. Each vector column
> row is 64 bytes (`dim × 2` for F16 storage).